# Subtask 1 — Ensemble & Post-processing (Notebook 4 of 5)

**Goal:** Squeeze the last points of Accuracy±1 by combining models and applying spatial post-processing.

**Techniques evaluated:**
1. Soft ensemble of pixel baseline + U-Net (tune weight α)
2. 3×3 median filter (reduces salt-and-pepper noise; safe for ordinal classes)
3. Test-time augmentation (flip left/right and up/down, average logits)

**HITL gate:** VB selects the configuration with highest val Acc±1 and enters it into Notebook 5's config cell.

## 0. Setup

In [ ]:
import csv
import json
import os
from pathlib import Path
from urllib.parse import urljoin
from urllib.request import urlopen

import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import median_filter
from tqdm.auto import tqdm
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import joblib

# ── REPO ROOT — works whether JupyterLab cwd is project root or notebooks/ ──
_cwd = Path(os.getcwd())
REPO_ROOT = _cwd if (_cwd / "scripts").exists() else _cwd.parent

# ── CONFIG — adjust to match Notebook 2 and 3 choices ───────────────────────
DATA_DIR         = None
LABEL_NAME       = "viticulture"
HF_BASE          = "https://huggingface.co/datasets/m-sakka/agripotential/resolve/main/"
CKPT_DIR         = REPO_ROOT / "results" / "subtask1" / "checkpoints"
OUT_DIR          = REPO_ROOT / "results" / "subtask1" / "ensemble"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TEMPORAL_OPTION  = "summary"   # must match Notebook 3 training
PIXEL_MODEL_NAME = "hgbc"      # best model from Notebook 2: 'hgbc' or 'etc'

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_BANDS = 10

CLASS_NAMES  = ["Very Low","Low","Medium","High","Very High","Unknown"]
CLASS_COLORS = ["#d73027","#fc8d59","#fee08b","#91cf60","#1a9850","#808080"]
print(f"Repo root: {REPO_ROOT}")
print(f"Device: {DEVICE}, temporal_option: {TEMPORAL_OPTION}")

## 1. Load Models & Metadata

In [ ]:
import rasterio
from rasterio.windows import Window

def ref(name):
    if DATA_DIR:
        return str(Path(DATA_DIR) / name)
    return urljoin(HF_BASE, name)

def read_csv(name):
    r = ref(name)
    if r.startswith("http"):
        with urlopen(r, timeout=60) as h:
            lines = [l.decode("utf-8") for l in h]
    else:
        lines = Path(r).read_text().splitlines()
    return list(csv.DictReader(lines))

metadata   = read_csv("metadata.csv")
val_rows   = read_csv("val.csv")

meta_df = pd.DataFrame(metadata)
meta_df["date"] = pd.to_datetime(meta_df[["year","month","day"]].astype(int))
meta_df = meta_df.sort_values("date").reset_index(drop=True)
sentinel_files = list(meta_df["filename"])
n_times = len(sentinel_files)
tf_months = list(meta_df["month"].astype(int))
SPRING_IDX = [i for i,m in enumerate(tf_months) if m in (3,4,5)]
SUMMER_IDX = [i for i,m in enumerate(tf_months) if m in (6,7,8)]
AUTUMN_IDX = [i for i,m in enumerate(tf_months) if m in (9,10,11)]
WINTER_IDX = [i for i,m in enumerate(tf_months) if m in (12,1,2)]

label_ref = ref(f"{LABEL_NAME}.tif")

# Load pixel model
pixel_model_path = CKPT_DIR / f"{PIXEL_MODEL_NAME}_pixel.pkl"
pixel_model = joblib.load(pixel_model_path)
print(f"Loaded pixel model: {pixel_model_path}")

# Load U-Net
unet_ckpt_path = CKPT_DIR / f"unet_{TEMPORAL_OPTION}_best.pt"
ckpt = torch.load(unet_ckpt_path, map_location=DEVICE)
n_channels = ckpt["n_channels"]
NORM_MEAN = np.array(ckpt["norm_mean"])
NORM_STD  = np.array(ckpt["norm_std"])
print(f"Loaded U-Net from epoch {ckpt['epoch']} (val Acc±1={ckpt['val_acc_pm1']:.4f})")

In [ ]:
# Re-define model and load weights
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.net(x)

class UNet(nn.Module):
    def __init__(self, in_channels, n_classes, base_ch=32):
        super().__init__()
        c = base_ch
        self.enc1=ConvBlock(in_channels,c); self.enc2=ConvBlock(c,c*2)
        self.enc3=ConvBlock(c*2,c*4);      self.bot=ConvBlock(c*4,c*8)
        self.up3=nn.ConvTranspose2d(c*8,c*4,2,stride=2); self.dec3=ConvBlock(c*8,c*4)
        self.up2=nn.ConvTranspose2d(c*4,c*2,2,stride=2); self.dec2=ConvBlock(c*4,c*2)
        self.up1=nn.ConvTranspose2d(c*2,c,2,stride=2);   self.dec1=ConvBlock(c*2,c)
        self.head=nn.Conv2d(c,n_classes,1); self.pool=nn.MaxPool2d(2)
    def forward(self,x):
        e1=self.enc1(x); e2=self.enc2(self.pool(e1)); e3=self.enc3(self.pool(e2))
        b=self.bot(self.pool(e3))
        d3=self.dec3(torch.cat([self.up3(b),e3],dim=1))
        d2=self.dec2(torch.cat([self.up2(d3),e2],dim=1))
        d1=self.dec1(torch.cat([self.up1(d2),e1],dim=1))
        return self.head(d1)

unet = UNet(in_channels=n_channels, n_classes=5).to(DEVICE)
unet.load_state_dict(ckpt["model_state"])
unet.eval()
print(f"U-Net ready ({n_channels} input channels)")

## 2. Per-Patch Prediction Utilities

In [ ]:
def accuracy_pm1(y_true, y_pred):
    return float(np.mean(np.abs(y_true.astype(int) - y_pred.astype(int)) <= 1))


def load_patch_stack(row_idx, col_idx, patch_size):
    """Return (T, B, H, W) float32 stack for one patch."""
    win = Window(col_idx, row_idx, patch_size, patch_size)
    frames = []
    for fname in sentinel_files:
        with rasterio.open(ref(fname)) as src:
            frames.append(src.read(window=win).astype(np.float32))
    return np.stack(frames, axis=0)


def stack_to_tensor(stack, temporal_option):
    T, B, H, W = stack.shape
    if temporal_option == "concat":
        x = stack.reshape(T*B, H, W)
    elif temporal_option == "summary":
        x = np.concatenate([stack.mean(0), stack.std(0)], axis=0)
    elif temporal_option == "seasonal":
        parts = []
        for idx in [SPRING_IDX, SUMMER_IDX, AUTUMN_IDX, WINTER_IDX]:
            parts.append(stack[idx].mean(0) if idx else np.zeros((B,H,W), dtype=np.float32))
        x = np.concatenate(parts, axis=0)
    t = torch.from_numpy(x)  # (C, H, W)
    mean_t = torch.tensor(NORM_MEAN[:, None, None], dtype=torch.float32)
    std_t  = torch.tensor(NORM_STD[:, None, None],  dtype=torch.float32)
    return ((t - mean_t) / std_t).unsqueeze(0)  # (1, C, H, W)


def pixel_features_from_stack(stack):
    T, B, H, W = stack.shape
    P = H * W
    pix = stack.reshape(T, B, P).transpose(2, 0, 1)  # (P, T, B)
    raw = stack.transpose(2,3,0,1).reshape(P, T*B)
    tmean = pix.mean(1); tstd = pix.std(1); tmin = pix.min(1); tmax = pix.max(1)
    parts = [raw, tmean, tstd, tmin, tmax]
    for idx in [SPRING_IDX, SUMMER_IDX, AUTUMN_IDX, WINTER_IDX]:
        parts.append(pix[:, idx, :].mean(1) if idx else np.zeros((P,B), dtype=np.float32))
    nir = pix[:,:,7]; red = pix[:,:,3]
    ndvi = (nir-red)/(nir+red+1e-6)
    parts += [ndvi.mean(1, keepdims=True), ndvi.std(1, keepdims=True)]
    return np.concatenate(parts, axis=1)


def predict_patch_unet(stack, tta=False):
    """Return softmax probabilities (5, H, W) with optional TTA."""
    H, W = stack.shape[2], stack.shape[3]
    x = stack_to_tensor(stack, TEMPORAL_OPTION).to(DEVICE)
    with torch.no_grad():
        logits = [unet(x)]
        if tta:
            logits.append(torch.flip(unet(torch.flip(x, [-1])), [-1]))
            logits.append(torch.flip(unet(torch.flip(x, [-2])), [-2]))
            logits.append(torch.flip(unet(torch.flip(x, [-1,-2])), [-1,-2]))
        avg_logit = torch.stack(logits, dim=0).mean(0)  # (1, 5, H, W)
        probs = F.softmax(avg_logit, dim=1)[0].cpu().numpy()  # (5, H, W)
    return probs


def predict_patch_pixel(stack):
    """Return class-probability matrix (5, H, W) from pixel model."""
    H, W = stack.shape[2], stack.shape[3]
    feats = pixel_features_from_stack(stack)
    probs = pixel_model.predict_proba(feats)  # (P, 5)
    return probs.T.reshape(5, H, W).astype(np.float32)


print("Utilities defined.")

## 3. Grid Search: Ensemble Weight α and Post-processing

In [ ]:
# Run a mini grid search over (α, use_median_filter, use_tta) on val.
# Only runs on the first VAL_LIMIT patches to keep it fast; increase if time allows.
VAL_LIMIT   = 100   # patches to evaluate; set to len(val_rows) for full eval
ALPHA_GRID  = [0.0, 0.2, 0.5, 0.8, 1.0]  # 0=pixel only, 1=U-Net only
MEDIAN_SIZES = [None, 3]      # None=no filter, 3=3×3
TTA_OPTIONS  = [False, True]

results = []
val_subset = val_rows[:VAL_LIMIT]

# Pre-compute both models' probabilities for each val patch (reuse across grid)
print(f"Pre-computing predictions for {len(val_subset)} val patches...")
patch_data = []  # list of (probs_pixel, probs_unet_base, probs_unet_tta, y_true)

with rasterio.open(label_ref) as lsrc:
    for patch in tqdm(val_subset):
        r, c, ps = int(patch["row"]), int(patch["col"]), int(patch["patch_size"])
        stack = load_patch_stack(r, c, ps)
        lbl = lsrc.read(1, window=Window(c, r, ps, ps)).astype(np.uint8)
        pp  = predict_patch_pixel(stack)
        pu  = predict_patch_unet(stack, tta=False)
        put = predict_patch_unet(stack, tta=True)
        patch_data.append((pp, pu, put, lbl))

print("Pre-computation complete. Running grid search...")

for alpha in ALPHA_GRID:
    for med_size in MEDIAN_SIZES:
        for use_tta in TTA_OPTIONS:
            all_pred, all_true = [], []
            for (pp, pu, put, lbl) in patch_data:
                p_unet = put if use_tta else pu
                ensemble = (1 - alpha) * pp + alpha * p_unet  # (5, H, W)
                pred = ensemble.argmax(axis=0).astype(np.uint8)  # (H, W)
                if med_size:
                    pred = median_filter(pred, size=med_size).astype(np.uint8)
                all_pred.append(pred.ravel())
                all_true.append(lbl.ravel())
            y_pred = np.concatenate(all_pred)
            y_true = np.concatenate(all_true)
            apm1  = accuracy_pm1(y_true, y_pred)
            exact = float(np.mean(y_pred == y_true))
            results.append({"alpha": alpha, "median_size": med_size,
                            "tta": use_tta, "acc_pm1": apm1, "exact": exact})

res_df = pd.DataFrame(results).sort_values("acc_pm1", ascending=False)
print("\n=== Grid search results (top 10) ===")
print(res_df.head(10).to_string(index=False))
best_row = res_df.iloc[0]
print(f"\nBest config: alpha={best_row['alpha']}, median={best_row['median_size']}, "
      f"tta={best_row['tta']}  →  Acc±1={best_row['acc_pm1']:.4f}")
res_df.to_csv(OUT_DIR / "ensemble_grid_results.csv", index=False)
print(f"Saved → {OUT_DIR}/ensemble_grid_results.csv")

## 4. Visualise Best Config vs Baseline

In [ ]:
import matplotlib.colors as mcolors

BEST_ALPHA    = float(best_row["alpha"])
BEST_MEDIAN   = best_row["median_size"]
BEST_TTA      = bool(best_row["tta"])

cmap = mcolors.ListedColormap(CLASS_COLORS)
norm = mcolors.BoundaryNorm([-0.5,0.5,1.5,2.5,3.5,4.5,5.5], cmap.N)

fig, axes = plt.subplots(5, 3, figsize=(10, 17))
with rasterio.open(label_ref) as lsrc:
    for row_i, patch in enumerate(val_rows[:5]):
        r, c, ps = int(patch["row"]), int(patch["col"]), int(patch["patch_size"])
        lbl = lsrc.read(1, window=Window(c, r, ps, ps)).astype(np.uint8)
        pp, pu, put, _ = patch_data[row_i]
        # Pixel-only
        pred_pix = pp.argmax(0).astype(np.uint8)
        # Best ensemble
        p_unet = put if BEST_TTA else pu
        ens = ((1-BEST_ALPHA)*pp + BEST_ALPHA*p_unet).argmax(0).astype(np.uint8)
        if BEST_MEDIAN:
            ens = median_filter(ens, size=BEST_MEDIAN).astype(np.uint8)

        for col_i, (img, title) in enumerate([
            (lbl, "Ground truth"),
            (pred_pix, f"Pixel Acc±1={accuracy_pm1(lbl.ravel(), pred_pix.ravel()):.3f}"),
            (ens,  f"Ensemble Acc±1={accuracy_pm1(lbl.ravel(), ens.ravel()):.3f}"),
        ]):
            axes[row_i, col_i].imshow(img, cmap=cmap, norm=norm, interpolation="nearest")
            axes[row_i, col_i].set_title(title, fontsize=8)
            axes[row_i, col_i].axis("off")

plt.suptitle(f"α={BEST_ALPHA}  median={BEST_MEDIAN}  tta={BEST_TTA}", fontsize=11)
plt.tight_layout()
plt.savefig(OUT_DIR / "ensemble_comparison.png", dpi=120)
plt.show()
print(f"Saved → {OUT_DIR}/ensemble_comparison.png")

## 5. Save Best Config for Notebook 5

In [ ]:
config = {
    "alpha": BEST_ALPHA,
    "median_filter_size": int(BEST_MEDIAN) if BEST_MEDIAN else None,
    "use_tta": BEST_TTA,
    "temporal_option": TEMPORAL_OPTION,
    "pixel_model_name": PIXEL_MODEL_NAME,
    "pixel_model_path": str(CKPT_DIR / f"{PIXEL_MODEL_NAME}_pixel.pkl"),
    "unet_ckpt_path": str(unet_ckpt_path),
    "val_acc_pm1": float(best_row["acc_pm1"]),
    "val_exact":   float(best_row["exact"]),
}
(OUT_DIR / "best_config.json").write_text(json.dumps(config, indent=2))
print("Saved best_config.json:")
print(json.dumps(config, indent=2))
print("\n→ Copy these values into the CONFIG cell at the top of Notebook 5.")

## 6. HITL Handoff

Review `ensemble_comparison.png` and the grid search table, then:

1. Is the best Acc±1 significantly better than both individual models? If not, just use the U-Net alone in Notebook 5.
2. Does the 3×3 median filter help visually (smoother masks without blurring true boundaries)? If so, keep it.
3. Copy the values from `best_config.json` into Notebook 5's CONFIG cell before running inference.